# P2 — Vietnamese News Classification

## Phase 3 — Machine Learning Model Benchmark

**Dataset:** UVN-1 v1.0.0  
**Samples:** 3,246  
**Classes:** 13  
**Primary Metric:** Macro-F1  
**Random Seed:** 42

### Objective

Benchmark three classical Machine Learning classifiers:

- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM

For Logistic Regression and Linear SVM, compare:

- `class_weight=None`
- `class_weight="balanced"`

### Data Leakage Protocol

- TRAIN: fit TF-IDF + classifier
- VALIDATION: model selection and comparison
- TEST: 🔒 LOCKED until final evaluation
- No model selection after test evaluation

### Phase 3 Status

This notebook is reconstructed from the frozen Phase 3 metadata and
model artifacts. The original notebook file was lost, so this notebook
documents and validates the frozen experiment rather than retraining it.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd


# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    raise RuntimeError(
        f"Project root not detected correctly: {PROJECT_ROOT}"
    )

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"


# ---------------------------------------------------------
# Phase 3 artifacts
# ---------------------------------------------------------

METADATA_PATH = (
    PROCESSED_DIR / "phase3_model_benchmark_metadata.json"
)

TFIDF_PATH = MODELS_DIR / "p2_tfidf_vectorizer.joblib"

MODEL_PATH = MODELS_DIR / "p2_linear_svm_balanced.joblib"


# ---------------------------------------------------------
# Basic artifact existence check
# ---------------------------------------------------------

required_files = [
    METADATA_PATH,
    TFIDF_PATH,
    MODEL_PATH,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required Phase 3 artifact not found: {path}"
        )


# ---------------------------------------------------------
# Load frozen metadata and model artifacts
# ---------------------------------------------------------

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    phase3_metadata = json.load(f)

tfidf_vectorizer = joblib.load(TFIDF_PATH)
selected_model = joblib.load(MODEL_PATH)


# ---------------------------------------------------------
# Display basic information
# ---------------------------------------------------------

print("PHASE 3 ARTIFACTS LOADED")
print("-" * 50)

print(f"Dataset version : {phase3_metadata['dataset_version']}")
print(f"Classes         : {phase3_metadata['number_of_classes']}")
print(f"Primary metric  : {phase3_metadata['primary_metric']}")

print(f"TF-IDF artifact : {TFIDF_PATH.name}")
print(f"Model artifact  : {MODEL_PATH.name}")

print()
print("Selected model:")
print(f"  Model         : {phase3_metadata['selected_model']['model']}")
print(
    f"  class_weight  : "
    f"{phase3_metadata['selected_model']['class_weight']}"
)
print(
    f"  Validation F1 : "
    f"{phase3_metadata['selected_model']['macro_f1']:.4f}"
)

print()
print("Artifact loading: PASS")

PHASE 3 ARTIFACTS LOADED
--------------------------------------------------
Dataset version : uvn1-v1.0.0
Classes         : 13
Primary metric  : macro_f1
TF-IDF artifact : p2_tfidf_vectorizer.joblib
Model artifact  : p2_linear_svm_balanced.joblib

Selected model:
  Model         : Linear SVM — Balanced
  class_weight  : balanced
  Validation F1 : 0.7136

Artifact loading: PASS


## Section 1 — Dataset & TF-IDF Contract

This section validates the frozen Phase 3 metadata and TF-IDF artifact.

### Frozen Dataset Contract

- Dataset version: `uvn1-v1.0.0`
- Total samples: **3,246**
- Classes: **13**
- Train: **2,272**
- Validation: **487**
- Test: **487**

### Frozen TF-IDF Contract

- `ngram_range = (1, 1)`
- `min_df = 5`
- `sublinear_tf = False`
- `lowercase = False`

The validation below reads the actual metadata structure instead of assuming
a nested dictionary layout.

The Phase 3 test set remains locked after its final evaluation.

In [5]:
# ---------------------------------------------------------
# Section 1 — Validate Phase 3 dataset & TF-IDF contract
# ---------------------------------------------------------

checks = []


def check(name: str, condition: bool):
    status = "PASS" if condition else "FAIL"
    checks.append((name, condition))
    print(f"{status}: {name}")


# ---------------------------------------------------------
# Phase 3 dataset contract
# ---------------------------------------------------------

check(
    "Dataset",
    phase3_metadata["dataset"] == "UVN-1",
)

check(
    "Dataset version",
    phase3_metadata["dataset_version"] == "uvn1-v1.0.0",
)

check(
    "Number of classes",
    phase3_metadata["number_of_classes"] == 13,
)

check(
    "Class list length",
    len(phase3_metadata["classes"]) == 13,
)

check(
    "Primary metric",
    phase3_metadata["primary_metric"] == "macro_f1",
)

check(
    "Random seed",
    phase3_metadata["random_seed"] == 42,
)


# ---------------------------------------------------------
# Phase 3 TF-IDF contract from frozen metadata
# ---------------------------------------------------------

tfidf_config = phase3_metadata["tfidf_config"]

check(
    "Phase 3 TF-IDF ngram_range",
    tuple(tfidf_config["ngram_range"]) == (1, 1),
)

check(
    "Phase 3 TF-IDF min_df",
    tfidf_config["min_df"] == 5,
)

check(
    "Phase 3 TF-IDF sublinear_tf",
    tfidf_config["sublinear_tf"] is False,
)

check(
    "Phase 3 TF-IDF lowercase",
    tfidf_config["lowercase"] is False,
)


# ---------------------------------------------------------
# Packaged Phase 5 artifact information
# ---------------------------------------------------------
# IMPORTANT:
# p2_tfidf_vectorizer.joblib is the packaged production
# vectorizer from Phase 5, not the original Phase 3
# benchmark vectorizer.
#
# Therefore it must NOT be compared against the Phase 3
# TF-IDF configuration above.

print()
print("Packaged production vectorizer:")
print(f"  ngram_range  : {tfidf_vectorizer.ngram_range}")
print(f"  min_df       : {tfidf_vectorizer.min_df}")
print(f"  sublinear_tf : {tfidf_vectorizer.sublinear_tf}")
print(f"  lowercase    : {tfidf_vectorizer.lowercase}")

print()
print("Artifact role:")
print("  Phase 3 metadata : frozen benchmark contract")
print("  Loaded vectorizer: Phase 5 production artifact")
print("  Loaded model     : Phase 5 packaged selected model")


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

print()
print("-" * 50)

all_passed = all(condition for _, condition in checks)

if not all_passed:
    failed = [
        name
        for name, condition in checks
        if not condition
    ]

    raise AssertionError(
        "Phase 3 contract validation failed: "
        + ", ".join(failed)
    )

print("SECTION 1 DATASET & TF-IDF CONTRACT: PASS")

PASS: Dataset
PASS: Dataset version
PASS: Number of classes
PASS: Class list length
PASS: Primary metric
PASS: Random seed
PASS: Phase 3 TF-IDF ngram_range
PASS: Phase 3 TF-IDF min_df
PASS: Phase 3 TF-IDF sublinear_tf
PASS: Phase 3 TF-IDF lowercase

Packaged production vectorizer:
  ngram_range  : (1, 2)
  min_df       : 1
  sublinear_tf : False
  lowercase    : False

Artifact role:
  Phase 3 metadata : frozen benchmark contract
  Loaded vectorizer: Phase 5 production artifact
  Loaded model     : Phase 5 packaged selected model

--------------------------------------------------
SECTION 1 DATASET & TF-IDF CONTRACT: PASS


## Section 2 — Model Benchmark Results

The Phase 3 benchmark compared:

1. Multinomial Naive Bayes
2. Logistic Regression — `class_weight=None`
3. Logistic Regression — `class_weight="balanced"`
4. Linear SVM — `class_weight=None`
5. Linear SVM — `class_weight="balanced"`

The primary selection metric is **Macro-F1** because the UVN-1 dataset is highly imbalanced.

All values below are reconstructed from the frozen Phase 3 metadata.
No model is retrained in this notebook.

In [7]:
# ---------------------------------------------------------
# Section 2 — Frozen Phase 3 benchmark results
# ---------------------------------------------------------

benchmark_results = phase3_metadata["benchmark_results"]

rows = []

for result in benchmark_results:
    rows.append(
        {
            "Model": result["model"],
            "Class Weight": result.get("class_weight", "N/A"),
            "Accuracy": result["accuracy"],
            "Precision Macro": result["precision_macro"],
            "Recall Macro": result["recall_macro"],
            "Macro-F1": result["macro_f1"],
            "Weighted-F1": result["weighted_f1"],
        }
    )

benchmark_df = pd.DataFrame(rows)

# Sort by primary metric: Macro-F1
benchmark_df = benchmark_df.sort_values(
    by="Macro-F1",
    ascending=False,
).reset_index(drop=True)


# ---------------------------------------------------------
# Format numeric columns without pandas Styler
# ---------------------------------------------------------

numeric_columns = [
    "Accuracy",
    "Precision Macro",
    "Recall Macro",
    "Macro-F1",
    "Weighted-F1",
]

benchmark_display = benchmark_df.copy()

for column in numeric_columns:
    benchmark_display[column] = benchmark_display[column].map(
        lambda value: f"{value:.4f}"
    )


# ---------------------------------------------------------
# Display benchmark table
# ---------------------------------------------------------

print("PHASE 3 MODEL BENCHMARK")
print("-" * 80)

display(benchmark_display)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

assert len(benchmark_df) == 5, (
    f"Expected 5 benchmark results, "
    f"got {len(benchmark_df)}"
)

assert benchmark_df["Macro-F1"].notna().all()

assert (
    benchmark_df.iloc[0]["Model"]
    == "Linear SVM — Balanced"
)

assert abs(
    benchmark_df.iloc[0]["Macro-F1"] - 0.71359563750333
) < 1e-10


print()
print("Benchmark results loaded:", len(benchmark_df))
print(
    f"Best model: {benchmark_df.iloc[0]['Model']}"
)
print(
    f"Best validation Macro-F1: "
    f"{benchmark_df.iloc[0]['Macro-F1']:.4f}"
)
print("SECTION 2 BENCHMARK RESULTS: PASS")

PHASE 3 MODEL BENCHMARK
--------------------------------------------------------------------------------


,Model,Class Weight,Accuracy,Precision Macro,Recall Macro,Macro-F1,Weighted-F1
0,Linear SVM — Balanced,balanced,0.8501,0.7109,0.7216,0.7136,0.8432
1,Logistic Regression — Balanced,balanced,0.8234,0.6982,0.7385,0.7118,0.8251
2,Linear SVM — None,None,0.8563,0.7467,0.6874,0.7064,0.8447
3,Logistic Regression — None,None,0.8296,0.7620,0.6256,0.6673,0.8109
4,MultinomialNB,Not Applicable,0.6653,0.3933,0.2852,0.2870,0.5803



Benchmark results loaded: 5
Best model: Linear SVM — Balanced
Best validation Macro-F1: 0.7136
SECTION 2 BENCHMARK RESULTS: PASS


## Section 3 — Model Selection

The final Phase 3 model is selected using the validation set and the
primary metric **Macro-F1**.

### Selected Model

**Linear SVM — Balanced**

- Class weight: `balanced`
- Validation Accuracy: **0.8501**
- Validation Macro-F1: **0.7136**
- Validation Weighted-F1: **0.8432**

### Selection Rule

The selected model has the highest validation Macro-F1 among the five
benchmarked configurations.

The test set was not used for model selection.

In [9]:
# ---------------------------------------------------------
# Section 3 — Validate selected model
# ---------------------------------------------------------

selected_metadata = phase3_metadata["selected_model"]

print("SELECTED MODEL")
print("-" * 60)

print(f"Model          : {selected_metadata['model']}")
print(f"Class weight   : {selected_metadata['class_weight']}")
print(f"Validation Acc : {selected_metadata['accuracy']:.4f}")
print(f"Validation F1  : {selected_metadata['macro_f1']:.4f}")


# ---------------------------------------------------------
# Find selected model in frozen benchmark results
# ---------------------------------------------------------

selected_benchmark = None

for result in phase3_metadata["benchmark_results"]:
    if (
        result["model"] == selected_metadata["model"]
        and result.get("class_weight")
        == selected_metadata["class_weight"]
    ):
        selected_benchmark = result
        break


assert selected_benchmark is not None, (
    "Selected model was not found in benchmark results."
)


# ---------------------------------------------------------
# Validate selected model against benchmark
# ---------------------------------------------------------

assert abs(
    selected_metadata["accuracy"]
    - selected_benchmark["accuracy"]
) < 1e-10

assert abs(
    selected_metadata["macro_f1"]
    - selected_benchmark["macro_f1"]
) < 1e-10


# ---------------------------------------------------------
# Validate that it is the highest Macro-F1
# ---------------------------------------------------------

best_macro_f1 = max(
    result["macro_f1"]
    for result in phase3_metadata["benchmark_results"]
)

assert abs(
    selected_metadata["macro_f1"]
    - best_macro_f1
) < 1e-10


# ---------------------------------------------------------
# Validate frozen selection
# ---------------------------------------------------------

assert selected_metadata["model"] == "Linear SVM — Balanced"

assert selected_metadata["class_weight"] == "balanced"

assert abs(
    selected_metadata["accuracy"]
    - 0.8501026694045175
) < 1e-10

assert abs(
    selected_metadata["macro_f1"]
    - 0.71359563750333
) < 1e-10


# ---------------------------------------------------------
# Display additional benchmark metric
# ---------------------------------------------------------

print(
    f"Validation W-F1: "
    f"{selected_benchmark['weighted_f1']:.4f}"
)

print()
print("Selection rule : Highest validation Macro-F1")
print("Selected model : Linear SVM — Balanced")
print("Selection validation: PASS")
print("SECTION 3 MODEL SELECTION: PASS")

SELECTED MODEL
------------------------------------------------------------
Model          : Linear SVM — Balanced
Class weight   : balanced
Validation Acc : 0.8501
Validation F1  : 0.7136
Validation W-F1: 0.8432

Selection rule : Highest validation Macro-F1
Selected model : Linear SVM — Balanced
Selection validation: PASS
SECTION 3 MODEL SELECTION: PASS


## Section 4 — Per-Class Validation Metrics

The selected Linear SVM — Balanced model is evaluated per class on the
validation set.

Metrics:

- Precision
- Recall
- F1-score
- Support

This section helps identify classes where the model performs well or
struggles despite the overall Macro-F1 score.

The metrics are reconstructed from the frozen Phase 3 metadata.

In [12]:
# ---------------------------------------------------------
# Section 4 — Per-class validation metrics
# ---------------------------------------------------------

per_class_metrics = phase3_metadata[
    "per_class_validation_metrics"
]

print("PER-CLASS METADATA STRUCTURE")
print("-" * 80)

print("Type:", type(per_class_metrics).__name__)
print("Number of classes:", len(per_class_metrics))

print()
print("First record:")
print(per_class_metrics[0])


# ---------------------------------------------------------
# Convert frozen metadata to DataFrame
# ---------------------------------------------------------

rows = []

for metrics in per_class_metrics:
    rows.append(
        {
            "Class": metrics["class"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"],
            "Support": metrics["support"],
        }
    )

per_class_df = pd.DataFrame(rows)


# ---------------------------------------------------------
# Sort by F1 — weakest first
# ---------------------------------------------------------

per_class_df = per_class_df.sort_values(
    by="F1",
    ascending=True,
).reset_index(drop=True)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print()
print("PER-CLASS VALIDATION METRICS")
print("-" * 80)

per_class_display = per_class_df.copy()

for column in ["Precision", "Recall", "F1"]:
    per_class_display[column] = per_class_display[column].map(
        lambda value: f"{value:.4f}"
    )

display(per_class_display)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

assert len(per_class_df) == 13, (
    f"Expected 13 classes, got {len(per_class_df)}"
)

assert per_class_df["Class"].nunique() == 13

assert per_class_df["F1"].notna().all()


# ---------------------------------------------------------
# Strongest / weakest classes
# ---------------------------------------------------------

strongest = per_class_df.iloc[-1]
weakest = per_class_df.iloc[0]

print()
print(
    f"Strongest class : {strongest['Class']} "
    f"(F1={strongest['F1']:.4f})"
)

print(
    f"Weakest class   : {weakest['Class']} "
    f"(F1={weakest['F1']:.4f})"
)

print()
print("Per-class metrics loaded:", len(per_class_df))
print("SECTION 4 PER-CLASS VALIDATION: PASS")

PER-CLASS METADATA STRUCTURE
--------------------------------------------------------------------------------
Type: list
Number of classes: 13

First record:
{'class': 'Công đoàn', 'precision': 0.8333333333333334, 'recall': 0.8333333333333334, 'f1': 0.8333333333333334, 'support': 6}

PER-CLASS VALIDATION METRICS
--------------------------------------------------------------------------------


,Class,Precision,Recall,F1,Support
0,Xã hội,0.0000,0.0000,0.0000,5
1,Thời sự,0.4211,0.3200,0.3636,25
2,Đời sống,0.6875,0.5000,0.5789,22
3,Pháp luật,0.5909,0.5909,0.5909,22
4,Xe,0.7500,0.7500,0.7500,4
5,Thế giới,0.7500,0.8000,0.7742,15
6,Sức khỏe,0.7667,0.8214,0.7931,28
7,Công đoàn,0.8333,0.8333,0.8333,6
8,Thể thao,0.8000,0.9524,0.8696,21
9,Khoa học,0.8861,0.8974,0.8917,78



Strongest class : Giáo dục (F1=0.9565)
Weakest class   : Xã hội (F1=0.0000)

Per-class metrics loaded: 13
SECTION 4 PER-CLASS VALIDATION: PASS


## Section 5 — Locked Test Evaluation

The Phase 3 test set was evaluated once after model selection.

### Test Set Policy

- Test samples: **487**
- Test status: **FINAL**
- Evaluation count: **1**
- Selection after test: **False**
- Tuning after test: **False**

The test set is permanently locked after this evaluation.

### Final Test Metrics

The frozen Phase 3 results are:

- Accuracy
- Macro Precision
- Macro Recall
- Macro-F1
- Weighted-F1

These values are reconstructed directly from the frozen Phase 3 metadata.

**Important:** This notebook does not retrain the model or re-run the test set.
The frozen Phase 3 test evaluation is the authoritative result.

In [15]:
# ---------------------------------------------------------
# Section 5 — Locked Test Evaluation
# ---------------------------------------------------------

test_set_metadata = phase3_metadata["test_set"]
final_evaluation = phase3_metadata["final_evaluation"]

test_metrics = final_evaluation["test"]
test_model = final_evaluation["model"]


# ---------------------------------------------------------
# Test-set lock validation
# ---------------------------------------------------------

print("LOCKED TEST SET VALIDATION")
print("-" * 80)

print(f"Status                : {test_set_metadata['status']}")
print(f"Evaluated             : {test_set_metadata['evaluated']}")
print(f"Evaluation count      : {test_set_metadata['evaluation_count']}")
print(f"Selection after test  : {test_set_metadata['selection_after_test']}")
print(f"Tuning after test     : {test_set_metadata['tuning_after_test']}")

assert test_set_metadata["status"] == "FINAL"
assert test_set_metadata["evaluated"] is True
assert test_set_metadata["evaluation_count"] == 1
assert test_set_metadata["selection_after_test"] is False
assert test_set_metadata["tuning_after_test"] is False

print()
print("Test-set lock: PASS")


# ---------------------------------------------------------
# Final frozen test evaluation
# ---------------------------------------------------------

print()
print("FINAL TEST EVALUATION")
print("-" * 80)

print(f"Model               : {test_model['name']}")
print(f"Class weight        : {test_model['class_weight']}")
print(f"Test samples        : {test_metrics['samples']}")
print(f"Accuracy            : {test_metrics['accuracy']:.4f}")
print(f"Precision Macro     : {test_metrics['precision_macro']:.4f}")
print(f"Recall Macro        : {test_metrics['recall_macro']:.4f}")
print(f"Macro-F1            : {test_metrics['macro_f1']:.4f}")
print(f"Weighted-F1         : {test_metrics['weighted_f1']:.4f}")


# ---------------------------------------------------------
# Validate frozen test metrics
# ---------------------------------------------------------

assert test_metrics["samples"] == 487

assert abs(
    test_metrics["accuracy"] - 0.8069815195071869
) < 1e-10

assert abs(
    test_metrics["precision_macro"] - 0.6853418453373575
) < 1e-10

assert abs(
    test_metrics["recall_macro"] - 0.7132097156274195
) < 1e-10

assert abs(
    test_metrics["macro_f1"] - 0.6871066933119706
) < 1e-10

assert abs(
    test_metrics["weighted_f1"] - 0.8004299483355166
) < 1e-10


# ---------------------------------------------------------
# Validate model identity
# ---------------------------------------------------------

assert test_model["name"] == "Linear SVM"
assert test_model["class_weight"] == "balanced"
assert test_model["random_state"] == 42
assert test_model["max_iter"] == 5000


# ---------------------------------------------------------
# Validate validation → test relationship
# ---------------------------------------------------------

validation_macro_f1 = final_evaluation["validation"]["macro_f1"]
test_macro_f1 = test_metrics["macro_f1"]

macro_f1_difference = test_macro_f1 - validation_macro_f1

print()
print(
    f"Validation Macro-F1 : {validation_macro_f1:.4f}"
)

print(
    f"Test Macro-F1       : {test_macro_f1:.4f}"
)

print(
    f"Val → Test difference: {macro_f1_difference:.4f}"
)


# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------

print()
print("Selected model : Linear SVM — Balanced")
print("Test evaluation: FROZEN")
print("No post-test tuning: PASS")
print("SECTION 5 LOCKED TEST EVALUATION: PASS")

LOCKED TEST SET VALIDATION
--------------------------------------------------------------------------------
Status                : FINAL
Evaluated             : True
Evaluation count      : 1
Selection after test  : False
Tuning after test     : False

Test-set lock: PASS

FINAL TEST EVALUATION
--------------------------------------------------------------------------------
Model               : Linear SVM
Class weight        : balanced
Test samples        : 487
Accuracy            : 0.8070
Precision Macro     : 0.6853
Recall Macro        : 0.7132
Macro-F1            : 0.6871
Weighted-F1         : 0.8004

Validation Macro-F1 : 0.7136
Test Macro-F1       : 0.6871
Val → Test difference: -0.0265

Selected model : Linear SVM — Balanced
Test evaluation: FROZEN
No post-test tuning: PASS
SECTION 5 LOCKED TEST EVALUATION: PASS


## Section 6 — Test Confusion Matrix

The confusion matrix below is the frozen confusion matrix from the final
Phase 3 test evaluation.

### Interpretation

- Rows represent the **true classes**.
- Columns represent the **predicted classes**.
- The diagonal contains correct predictions.
- Off-diagonal values represent classification errors.

The matrix contains all **13 UVN-1 classes** and covers the complete
487-sample test set.

This section only reconstructs the frozen result.
No model inference or test-set re-evaluation is performed.

In [16]:
# ---------------------------------------------------------
# Section 6 — Test Confusion Matrix
# ---------------------------------------------------------

test_confusion_matrix = final_evaluation[
    "test_confusion_matrix"
]

classes = phase3_metadata["classes"]


# ---------------------------------------------------------
# Validate matrix structure
# ---------------------------------------------------------

print("TEST CONFUSION MATRIX VALIDATION")
print("-" * 80)

print(f"Number of classes : {len(classes)}")
print(
    f"Matrix shape      : "
    f"{len(test_confusion_matrix)} x "
    f"{len(test_confusion_matrix[0])}"
)

assert len(classes) == 13

assert len(test_confusion_matrix) == 13

assert all(
    len(row) == 13
    for row in test_confusion_matrix
)

assert all(
    isinstance(value, int)
    for row in test_confusion_matrix
    for value in row
)


# ---------------------------------------------------------
# Convert to DataFrame
# ---------------------------------------------------------

confusion_df = pd.DataFrame(
    test_confusion_matrix,
    index=classes,
    columns=classes,
)

confusion_df.index.name = "True \\ Predicted"


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print()
print("TEST CONFUSION MATRIX")
print("-" * 80)

display(confusion_df)


# ---------------------------------------------------------
# Validate total samples
# ---------------------------------------------------------

matrix_total = sum(
    sum(row)
    for row in test_confusion_matrix
)

print()
print(f"Matrix total samples : {matrix_total}")
print(f"Expected test samples: {test_metrics['samples']}")

assert matrix_total == test_metrics["samples"]


# ---------------------------------------------------------
# Diagonal / correct predictions
# ---------------------------------------------------------

correct_predictions = sum(
    test_confusion_matrix[i][i]
    for i in range(len(classes))
)

print(
    f"Correct predictions  : {correct_predictions}"
)

print(
    f"Incorrect predictions: "
    f"{matrix_total - correct_predictions}"
)


# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

print()
print("Matrix dimensions: PASS")
print("Matrix sample count: PASS")
print("SECTION 6 TEST CONFUSION MATRIX: PASS")

TEST CONFUSION MATRIX VALIDATION
--------------------------------------------------------------------------------
Number of classes : 13
Matrix shape      : 13 x 13

TEST CONFUSION MATRIX
--------------------------------------------------------------------------------


,Công đoàn,Giáo dục,Giải trí,Khoa học,Kinh doanh,Pháp luật,Sức khỏe,Thế giới,Thể thao,Thời sự,Xe,Xã hội,Đời sống
True \ Predicted,,,,,,,,,,,,,
Công đoàn,3,1,0,0,0,0,0,0,0,2,0,0,0
Giáo dục,0,74,0,2,0,0,1,0,0,0,0,0,2
Giải trí,0,0,12,0,1,0,0,0,0,0,0,0,2
Khoa học,0,2,0,72,2,0,0,0,1,0,0,0,0
Kinh doanh,3,1,2,14,138,2,0,2,0,0,1,0,3
Pháp luật,0,1,0,0,1,19,0,0,0,0,1,0,0
Sức khỏe,0,1,0,1,0,0,25,0,0,0,0,0,1
Thế giới,0,0,0,3,1,1,3,5,0,1,0,0,2
Thể thao,0,0,0,0,0,0,0,0,22,0,0,0,0



Matrix total samples : 487
Expected test samples: 487
Correct predictions  : 393
Incorrect predictions: 94

Matrix dimensions: PASS
Matrix sample count: PASS
SECTION 6 TEST CONFUSION MATRIX: PASS


## Section 7 — Phase 3 Frozen Conclusion

Phase 3 established the final classical Machine Learning benchmark for
Vietnamese news classification on the UVN-1 dataset.

### Final Selected Model

**Linear SVM — Balanced**

Validation:

- Accuracy: **0.8501**
- Macro-F1: **0.7136**
- Weighted-F1: **0.8432**

Final locked test evaluation:

- Accuracy: **0.8070**
- Macro-F1: **0.6871**
- Weighted-F1: **0.8004**

### Model Selection

The selected model achieved the highest validation Macro-F1 among:

- Multinomial Naive Bayes
- Logistic Regression — `class_weight=None`
- Logistic Regression — `class_weight="balanced"`
- Linear SVM — `class_weight=None`
- Linear SVM — `class_weight="balanced"`

### Test Set Policy

The test set was evaluated once after model selection.

- Test evaluation count: **1**
- Selection after test: **False**
- Tuning after test: **False**

The Phase 3 test result is therefore frozen.

### Phase 3 Status

**PASS — Phase 3 Model Benchmark is frozen.**

The selected model and benchmark results are carried forward to the
subsequent packaging and API phases.

In [17]:
# ---------------------------------------------------------
# Section 7 — Final Phase 3 Integrity Check
# ---------------------------------------------------------

print("PHASE 3 FINAL INTEGRITY CHECK")
print("=" * 80)


# ---------------------------------------------------------
# Dataset contract
# ---------------------------------------------------------

assert phase3_metadata["dataset"] == "UVN-1"
assert phase3_metadata["dataset_version"] == "uvn1-v1.0.0"
assert phase3_metadata["number_of_classes"] == 13
assert len(phase3_metadata["classes"]) == 13

print("PASS: Dataset contract")


# ---------------------------------------------------------
# Benchmark contract
# ---------------------------------------------------------

benchmark_results = phase3_metadata["benchmark_results"]

assert len(benchmark_results) == 5

selected_model_metadata = phase3_metadata["selected_model"]

assert selected_model_metadata["model"] == "Linear SVM — Balanced"
assert selected_model_metadata["class_weight"] == "balanced"

best_macro_f1 = max(
    result["macro_f1"]
    for result in benchmark_results
)

assert abs(
    selected_model_metadata["macro_f1"] - best_macro_f1
) < 1e-10

print("PASS: Benchmark and model-selection contract")


# ---------------------------------------------------------
# Per-class validation contract
# ---------------------------------------------------------

per_class_validation = phase3_metadata[
    "per_class_validation_metrics"
]

assert len(per_class_validation) == 13

print("PASS: Validation per-class metrics")


# ---------------------------------------------------------
# Test-set lock contract
# ---------------------------------------------------------

test_set_metadata = phase3_metadata["test_set"]

assert test_set_metadata["status"] == "FINAL"
assert test_set_metadata["evaluated"] is True
assert test_set_metadata["evaluation_count"] == 1
assert test_set_metadata["selection_after_test"] is False
assert test_set_metadata["tuning_after_test"] is False

print("PASS: Test-set lock")


# ---------------------------------------------------------
# Final test evaluation contract
# ---------------------------------------------------------

final_evaluation = phase3_metadata["final_evaluation"]

assert final_evaluation["status"] == "FINAL"

assert final_evaluation["model"]["name"] == "Linear SVM"
assert final_evaluation["model"]["class_weight"] == "balanced"
assert final_evaluation["model"]["random_state"] == 42

test_metrics = final_evaluation["test"]

assert test_metrics["samples"] == 487

assert abs(
    test_metrics["accuracy"] - 0.8069815195071869
) < 1e-10

assert abs(
    test_metrics["macro_f1"] - 0.6871066933119706
) < 1e-10

assert abs(
    test_metrics["weighted_f1"] - 0.8004299483355166
) < 1e-10

print("PASS: Final test evaluation")


# ---------------------------------------------------------
# Confusion matrix contract
# ---------------------------------------------------------

confusion_matrix = final_evaluation[
    "test_confusion_matrix"
]

assert len(confusion_matrix) == 13
assert all(len(row) == 13 for row in confusion_matrix)

matrix_total = sum(
    sum(row)
    for row in confusion_matrix
)

assert matrix_total == test_metrics["samples"]

print("PASS: Test confusion matrix")


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

print()
print("=" * 80)
print("PHASE 3 MODEL BENCHMARK: FROZEN")
print("=" * 80)

print()
print("Selected model : Linear SVM — Balanced")
print(f"Validation F1 : {selected_model_metadata['macro_f1']:.4f}")
print(f"Test F1       : {test_metrics['macro_f1']:.4f}")
print(f"Test Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Test samples  : {test_metrics['samples']}")

print()
print("TEST SET STATUS")
print("-" * 80)
print("Evaluation count      : 1")
print("Selection after test  : False")
print("Tuning after test     : False")

print()
print("SECTION 7 FINAL INTEGRITY CHECK: PASS")
print("PHASE 3 NOTEBOOK RECONSTRUCTION: COMPLETE")

PHASE 3 FINAL INTEGRITY CHECK
PASS: Dataset contract
PASS: Benchmark and model-selection contract
PASS: Validation per-class metrics
PASS: Test-set lock
PASS: Final test evaluation
PASS: Test confusion matrix

PHASE 3 MODEL BENCHMARK: FROZEN

Selected model : Linear SVM — Balanced
Validation F1 : 0.7136
Test F1       : 0.6871
Test Accuracy : 0.8070
Test samples  : 487

TEST SET STATUS
--------------------------------------------------------------------------------
Evaluation count      : 1
Selection after test  : False
Tuning after test     : False

SECTION 7 FINAL INTEGRITY CHECK: PASS
PHASE 3 NOTEBOOK RECONSTRUCTION: COMPLETE


# PHASE 7 — PostgreSQL + News Storage

## SECTION 7.10 — DATABASE SERVICE HEALTH CHECK

### Objectives

- Verify the application can connect to PostgreSQL.
- Verify SQLAlchemy async database connectivity.
- Keep database schema management under Alembic.
- Ensure application code does not create tables automatically.
- Prepare the database layer for Phase 8 RSS ingestion.

### Rules

- PostgreSQL is the application database.
- SQLAlchemy handles database access.
- Alembic manages schema migrations.
- Application startup must never call `Base.metadata.create_all()`.
- Database connectivity failure must be detectable.
- ML Predictor remains independent from database implementation.

In [18]:
# ---------------------------------------------------------
# Section 7.10 — Database Service Health Check
# ---------------------------------------------------------

from src.db.database import (
    engine,
    check_db_connection,
)
from src.db.base import Base


# ---------------------------------------------------------
# Validate database engine
# ---------------------------------------------------------

print("DATABASE SERVICE VALIDATION")
print("-" * 80)

print(f"Engine type       : {type(engine).__name__}")
print(f"Engine URL driver : {engine.url.drivername}")

assert engine.url.drivername == "postgresql+asyncpg"

print("PASS: SQLAlchemy async PostgreSQL engine")


# ---------------------------------------------------------
# Validate ORM base
# ---------------------------------------------------------

assert Base is not None

print("PASS: SQLAlchemy Declarative Base")


# ---------------------------------------------------------
# Actual PostgreSQL connectivity
# ---------------------------------------------------------

db_ok = await check_db_connection()

assert db_ok is True

print("PASS: PostgreSQL connection")


# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------

print()
print("=" * 80)
print("SECTION 7.10 DATABASE SERVICE HEALTH CHECK: PASS")
print("=" * 80)

DATABASE SERVICE VALIDATION
--------------------------------------------------------------------------------
Engine type       : AsyncEngine
Engine URL driver : postgresql+asyncpg
PASS: SQLAlchemy async PostgreSQL engine
PASS: SQLAlchemy Declarative Base
PASS: PostgreSQL connection

SECTION 7.10 DATABASE SERVICE HEALTH CHECK: PASS
